# 4 delay pattern

Converted from a Marimo HTML export to a Jupyter Notebook (`.ipynb`).
Code order follows the original Marimo app.


In [ ]:
import marimo as mo
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path


# Task 4: Delay Pattern Analysis - Regularity (Bottleneck Detection)

**Objective**: Identify problematic segments that cause big delays/irregularity (bottlenecks) and analyze if there is a pattern.

This analysis focuses on:
- Identifying stops with consistently high delays
- Detecting delay propagation along the line
- Finding temporal patterns (by hour, day of week)
- Visualizing the line topology with delay statistics


In [ ]:
data__reg_path = Path("data/vehicle_delays_reg1.parquet")
data__punc_path = Path("data/vehicle_delays_punc1.parquet")

df_reg = pd.read_parquet(data__reg_path)
df_punc = pd.read_parquet(data__punc_path)

df = pd.concat([df_reg, df_punc], axis=0)

# Basic info
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Number of lines: {df['route_short_name'].nunique()}")
print(f"\nColumns: {df.columns.tolist()}")


In [ ]:
available_lines = sorted(df_reg['route_short_name'].unique())
line_selector = mo.ui.dropdown(
     options=available_lines,
     value=available_lines[0] if available_lines else None,
     label="Select Line (route_short_name):"
)
line_selector


In [ ]:
selected_line = line_selector.value
df_line = df[df['route_short_name'] == selected_line].copy()

# Get direction selector
available_directions = sorted(df_line['direction_id'].unique())


In [ ]:
df_line['stop_name_actual'].unique()


In [ ]:
direction_selector = mo.ui.dropdown(
     options=available_directions,
     value=available_directions[0] if available_directions else None,
     label="Select Direction:"
)
direction_selector


In [ ]:
selected_direction = direction_selector.value

df_analysis = df_line[df_line['direction_id'] == selected_direction].copy()

# Get trip_headsign for this line and direction
trip_headsign = df_analysis['trip_headsign'].mode()[0] if len(df_analysis) > 0 else "Unknown"

# Create stop_order: for each stop_id, get the most frequent max stop_sequence
# This gives us the canonical order of stops on the line
stop_order_mapping = df_analysis.groupby('stop_id').agg({
    'stop_sequence': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.max(),
    'stop_name_actual': 'first'
}).reset_index()

stop_order_mapping = stop_order_mapping.sort_values('stop_sequence').reset_index(drop=True)
stop_order_mapping['stop_order'] = range(1, len(stop_order_mapping) + 1)

# Merge stop_order back into df_analysis
df_analysis = df_analysis.merge(
    stop_order_mapping[['stop_id', 'stop_order']],
    on='stop_id',
    how='left'
)

print(f"\nLine {df_analysis['route_short_name'].iloc[0]} - Direction {selected_direction}")
print(f"Direction: {trip_headsign}")
print(f"Number of stops: {len(stop_order_mapping)}")
print(f"Number of records: {len(df_analysis)}")
print(f"\nStop Order Mapping:")
print(stop_order_mapping[['stop_order', 'stop_id', 'stop_name_actual', 'stop_sequence']].head(10))


In [ ]:
df_analysis[(df_analysis['trip_headsign'] == df_analysis['stop_name_actual'])]


In [ ]:
stop_order_mapping[['stop_order', 'stop_id', 'stop_name_actual', 'stop_sequence']]


In [ ]:
print(len(df_analysis['stop_name_actual'].unique()))
print(len(df_analysis['stop_id'].unique()))
print(len(df_analysis['stop_order'].unique()))
print(len(df_analysis['stop_sequence'].unique()))


## 1. Line Statistics Overview

Key metrics for identifying problematic segments:


In [ ]:
stop_stats = df_analysis.groupby('stop_id').agg({
    'delay_minutes': ['mean', 'median', 'std', 'min', 'max', 'count'],
    'headway_minutes': ['mean', 'std'],
    'stop_name_actual': 'first',
    'stop_order': 'first'
}).reset_index()

# Flatten column names
stop_stats.columns = ['stop_id', 'delay_mean', 'delay_median', 'delay_std',
                      'delay_min', 'delay_max', 'count',
                      'headway_mean', 'headway_std',
                      'stop_name', 'stop_order']

# Sort by stop_order
stop_stats = stop_stats.sort_values('stop_order')

# Calculate delay increase (bottleneck indicator)
stop_stats['delay_increase'] = stop_stats['delay_mean'].diff()
stop_stats['delay_increase_pct'] = (stop_stats['delay_mean'].diff() /
                                    stop_stats['delay_mean'].shift(1) * 100)

# Identify problematic stops (top 20% by mean delay)
delay_threshold = stop_stats['delay_mean'].quantile(0.80)
stop_stats['is_bottleneck'] = stop_stats['delay_mean'] >= delay_threshold

# Calculate coefficient of variation (CV) for irregularity
stop_stats['delay_cv'] = stop_stats['delay_std'] / (stop_stats['delay_mean'].abs() + 0.001)
stop_stats['headway_cv'] = stop_stats['headway_std'] / (stop_stats['headway_mean'] + 0.001)

# Display summary
summary_stats = pd.DataFrame({
    'Metric': [
        'Total Stops',
        'Problematic Stops (Bottlenecks)',
        'Average Delay (min)',
        'Max Delay (min)',
        'Avg Headway (min)',
        'Max Delay Increase Between Stops (min)'
    ],
    'Value': [
        len(stop_stats),
        stop_stats['is_bottleneck'].sum(),
        f"{stop_stats['delay_mean'].mean():.2f}",
        f"{stop_stats['delay_max'].max():.2f}",
        f"{stop_stats['headway_mean'].mean():.2f}",
        f"{stop_stats['delay_increase'].max():.2f}"
    ]
})

mo.md(f"""
### Summary Statistics

{mo.ui.table(summary_stats)}

**Bottleneck Criteria**: Stops with mean delay >= {delay_threshold:.2f} minutes (top 20%)
""")


In [ ]:
stop_stats


## 2. Line Topology Visualization with Delay Statistics

Visual representation of stops along the line with delay metrics:


In [ ]:
fig_topology = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Stop Sequence with Mean Delay', 'Delay Evolution Along Line'),
    column_widths=[0.3, 0.7],
    specs=[[{"type": "xy"}, {"type": "xy"}]]
)

# Left panel: Vertical line representation like metro map
y_positions = list(range(len(stop_stats)))

# Draw vertical line
for i in range(len(stop_stats) - 1):
    fig_topology.add_trace(
        go.Scatter(
            x=[0, 0],
            y=[y_positions[i], y_positions[i+1]],
            mode='lines',
            line=dict(color='#1f77b4', width=8),
            showlegend=False,
            hoverinfo='skip'
        ),
        row=1, col=1
    )

# Add stop circles colored by delay
colors = ['#d62728' if is_bottleneck else '#1f77b4'
          for is_bottleneck in stop_stats['is_bottleneck']]

fig_topology.add_trace(
    go.Scatter(
        x=[0] * len(stop_stats),
        y=y_positions,
        mode='markers+text',
        marker=dict(
            size=20,
            color=colors,
            line=dict(color='white', width=2)
        ),
        text=stop_stats['stop_name'].str[:15],  # Truncate long names
        textposition='middle right',
        textfont=dict(size=9),
        hovertemplate='<b>%{text}</b><br>' +
                     'Mean Delay: %{customdata[0]:.2f} min<br>' +
                     'Std Dev: %{customdata[1]:.2f} min<br>' +
                     'Order: %{customdata[2]}<br>' +
                     '<extra></extra>',
        customdata=stop_stats[['delay_mean', 'delay_std', 'stop_order']].values,
        showlegend=False
    ),
    row=1, col=1
)

# Right panel: Delay evolution with multiple metrics
fig_topology.add_trace(
    go.Scatter(
        x=stop_stats['stop_order'],
        y=stop_stats['delay_mean'],
        mode='lines+markers',
        name='Mean Delay',
        line=dict(color='#1f77b4', width=2),
        marker=dict(
            size=8,
            color=colors,
            line=dict(color='white', width=1)
        ),
        hovertemplate='<b>%{customdata[0]}</b><br>' +
                     'Stop Order: %{x}<br>' +
                     'Mean Delay: %{y:.2f} min<br>' +
                     '<extra></extra>',
        customdata=stop_stats[['stop_name']].values
    ),
    row=1, col=2
)

# Add error bars (std deviation)
fig_topology.add_trace(
    go.Scatter(
        x=stop_stats['stop_order'],
        y=stop_stats['delay_mean'] + stop_stats['delay_std'],
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip'
    ),
    row=1, col=2
)

fig_topology.add_trace(
    go.Scatter(
        x=stop_stats['stop_order'],
        y=stop_stats['delay_mean'] - stop_stats['delay_std'],
        mode='lines',
        line=dict(width=0),
        fillcolor='rgba(31, 119, 180, 0.2)',
        fill='tonexty',
        name='±1 Std Dev',
        hoverinfo='skip'
    ),
    row=1, col=2
)

# Add median delay line
fig_topology.add_trace(
    go.Scatter(
        x=stop_stats['stop_order'],
        y=stop_stats['delay_median'],
        mode='lines',
        name='Median Delay',
        line=dict(color='#ff7f0e', width=2, dash='dash')
    ),
    row=1, col=2
)

# Highlight bottleneck zones
bottleneck_stops = stop_stats[stop_stats['is_bottleneck']]
if len(bottleneck_stops) > 0:
    fig_topology.add_trace(
        go.Scatter(
            x=bottleneck_stops['stop_order'],
            y=bottleneck_stops['delay_mean'],
            mode='markers',
            name='Bottleneck',
            marker=dict(
                size=15,
                color='#d62728',
                symbol='star',
                line=dict(color='white', width=1)
            ),
            hovertemplate='<b>BOTTLENECK: %{customdata[0]}</b><br>' +
                         'Mean Delay: %{y:.2f} min<br>' +
                         '<extra></extra>',
            customdata=bottleneck_stops[['stop_name']].values
        ),
        row=1, col=2
    )

# Update layout
fig_topology.update_xaxes(title_text="", showticklabels=False, row=1, col=1)
fig_topology.update_yaxes(title_text="Stop Order", showticklabels=False, row=1, col=1)
fig_topology.update_xaxes(title_text="Stop Order", row=1, col=2)
fig_topology.update_yaxes(title_text="Delay (minutes)", row=1, col=2)

fig_topology.update_layout(
    height=800,
    title_text=f"Line {selected_line} - {trip_headsign}<br>Line Topology and Delay Analysis",
    hovermode='closest',
    showlegend=True,
    legend=dict(x=1.05, y=0.5)
)

fig_topology


## 3. Bottleneck Identification Table

Detailed statistics for problematic stops:


In [ ]:
bottleneck_table = stop_stats[stop_stats['is_bottleneck']].copy()
bottleneck_table = bottleneck_table[[
    'stop_order', 'stop_name', 'delay_mean', 'delay_median',
    'delay_std', 'delay_cv', 'delay_increase', 'headway_cv', 'count'
]].round(2)

bottleneck_table.columns = [
    'Order', 'Stop Name', 'Mean Delay', 'Median Delay',
    'Std Dev', 'CV Delay', 'Delay Increase', 'CV Headway', 'Observations'
]

print(f"""
### Bottleneck Stops
""")

mo.ui.table(
     bottleneck_table,
     selection=None,
     label="Bottleneck Stops (Top 20% by Mean Delay)"
)


## 4. Delay Patterns by Time of Day

Analyze how delays vary throughout the day:


In [ ]:
hourly_delay = df_analysis.groupby(['stop_id', 'arrival_hour_actual']).agg({
    'delay_minutes': 'mean',
    'stop_name_actual': 'first',
    'stop_order': 'first'
}).reset_index()

# Create heatmap
# Pivot to get stops x hours matrix
delay_pivot = hourly_delay.pivot_table(
    index='stop_id',
    columns='arrival_hour_actual',
    values='delay_minutes'
)

# Sort by stop_order
stop_id_order = stop_stats.sort_values('stop_order')['stop_id'].tolist()
delay_pivot = delay_pivot.reindex(stop_id_order)

# Get stop names in order
stop_names = [stop_stats[stop_stats['stop_id'] == sid]['stop_name'].iloc[0][:20]
              for sid in stop_id_order]

fig_heatmap = go.Figure(data=go.Heatmap(
    z=delay_pivot.values,
    x=delay_pivot.columns,
    y=stop_names,
    colorscale='RdYlGn_r',
    zmid=0,
    colorbar=dict(title="Delay (min)"),
    hovertemplate='Stop: %{y}<br>Hour: %{x}:00<br>Avg Delay: %{z:.2f} min<extra></extra>'
))

fig_heatmap.update_layout(
    title=f"Line {selected_line} - {trip_headsign}<br>Average Delay by Stop and Hour of Day",
    xaxis_title="Hour of Day",
    yaxis_title="Stop (in sequence order)",
    height=max(400, len(stop_names) * 20),
    yaxis=dict(tickfont=dict(size=8))
)

fig_heatmap


## 5. Delay Propagation Analysis

Analyze how delays propagate along the line:


In [ ]:
fig_propagation = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Delay Increase Between Consecutive Stops',
                   'Cumulative Delay Along Line'),
    vertical_spacing=0.12
)

# Delay increase between stops
fig_propagation.add_trace(
    go.Bar(
        x=stop_stats['stop_order'][1:],
        y=stop_stats['delay_increase'][1:],
        marker_color=['#d62728' if x > 0.5 else '#2ca02c'
                     for x in stop_stats['delay_increase'][1:]],
        hovertemplate='Stop %{x}<br>Delay Increase: %{y:.2f} min<br>' +
                     '<b>%{customdata}</b><extra></extra>',
        customdata=stop_stats['stop_name'][1:].values
    ),
    row=1, col=1
)

# Cumulative delay
cumulative_delay = stop_stats['delay_mean'].cumsum()
fig_propagation.add_trace(
    go.Scatter(
        x=stop_stats['stop_order'],
        y=cumulative_delay,
        mode='lines+markers',
        line=dict(color='#1f77b4', width=2),
        marker=dict(size=6),
        hovertemplate='Stop: %{customdata}<br>Cumulative Delay: %{y:.2f} min<extra></extra>',
        customdata=stop_stats['stop_name'].values
    ),
    row=2, col=1
)

fig_propagation.update_xaxes(title_text="Stop Order", row=1, col=1)
fig_propagation.update_yaxes(title_text="Delay Increase (min)", row=1, col=1)
fig_propagation.update_xaxes(title_text="Stop Order", row=2, col=1)
fig_propagation.update_yaxes(title_text="Cumulative Delay (min)", row=2, col=1)

fig_propagation.update_layout(
    height=700,
    title_text=f"Line {selected_line} - {trip_headsign}<br>Delay Propagation Analysis",
    showlegend=False,
    hovermode='closest'
)

fig_propagation


## 6. Peak Hours Analysis

Identify peak hours with highest delays:


In [ ]:
hourly_stats = df_analysis.groupby('arrival_hour_actual').agg({
    'delay_minutes': ['mean', 'median', 'std', 'count'],
    'headway_minutes': ['mean', 'std']
}).reset_index()

hourly_stats.columns = ['hour', 'delay_mean', 'delay_median', 'delay_std',
                        'count', 'headway_mean', 'headway_std']

# Identify peak hours (top 25% by delay)
peak_threshold = hourly_stats['delay_mean'].quantile(0.75)
hourly_stats['is_peak'] = hourly_stats['delay_mean'] >= peak_threshold

hourly_stats


## 7. Key Findings and Patterns

Summary of identified patterns:


In [ ]:
top_bottlenecks = stop_stats.nlargest(5, 'delay_mean')[['stop_name', 'delay_mean', 'delay_increase', 'stop_order']]
worst_hours = hourly_stats.nlargest(3, 'delay_mean')[['hour', 'delay_mean']]

# Find biggest delay jumps
biggest_jumps = stop_stats.nlargest(3, 'delay_increase')[['stop_name', 'stop_order', 'delay_increase']]

insights_df = pd.DataFrame({
    'Finding': [
        'Number of Bottleneck Stops',
        'Worst Bottleneck Stop',
        'Average Delay at Bottlenecks',
        'Peak Delay Hour',
        'Biggest Delay Jump Between Stops',
        'Overall Line Regularity'
    ],
    'Value': [
        f"{stop_stats['is_bottleneck'].sum()} stops",
        f"{top_bottlenecks.iloc[0]['stop_name']} ({top_bottlenecks.iloc[0]['delay_mean']:.2f} min avg)",
        f"{stop_stats[stop_stats['is_bottleneck']]['delay_mean'].mean():.2f} minutes",
        f"{int(worst_hours.iloc[0]['hour'])}:00 ({worst_hours.iloc[0]['delay_mean']:.2f} min)",
        f"Stop {int(biggest_jumps.iloc[0]['stop_order'])}: {biggest_jumps.iloc[0]['stop_name']} (+{biggest_jumps.iloc[0]['delay_increase']:.2f} min)",
        f"{'Poor' if stop_stats['delay_mean'].mean() > 2 else 'Moderate' if stop_stats['delay_mean'].mean() > 1 else 'Good'} (avg {stop_stats['delay_mean'].mean():.2f} min delay)"
    ]
})

mo.md(f"""
### Pattern Analysis Summary

{mo.ui.table(insights_df)}

### Top 5 Bottleneck Stops

{mo.ui.table(top_bottlenecks.round(2))}

### Interpretation

- **Bottleneck Identification**: {stop_stats['is_bottleneck'].sum()} stops show consistently high delays (>{stop_stats['delay_mean'].quantile(0.80):.2f} min)
- **Delay Propagation**: The biggest delay increase occurs at stop {int(biggest_jumps.iloc[0]['stop_order'])} ({biggest_jumps.iloc[0]['stop_name']}), suggesting this segment is a critical bottleneck
- **Temporal Pattern**: Peak delays occur around {int(worst_hours.iloc[0]['hour'])}:00, likely due to traffic/passenger load
""")


## 8. Export Results

Export bottleneck analysis for further investigation:


In [ ]:
export_data = stop_stats.copy()
export_data['line'] = selected_line
export_data['direction'] = selected_direction

# Save to CSV
output_file = f"data/bottleneck/line_{selected_line}_dir{selected_direction}.csv"
export_data.to_csv(output_file, index=False)

# Also save detailed trip-level data for bottleneck stops
bottleneck_stop_ids = stop_stats[stop_stats['is_bottleneck']]['stop_id'].tolist()
bottleneck_trips = df_analysis[df_analysis['stop_id'].isin(bottleneck_stop_ids)]

trip_output_file = f"data/bottleneck/bottleneck_trips_{selected_line}_dir{selected_direction}.csv"
bottleneck_trips.to_csv(trip_output_file, index=False)

print(f"✓ Bottleneck analysis exported to: {output_file}")
print(f"✓ Detailed trip data exported to: {trip_output_file}")
print(f"  Total bottleneck stops: {len(bottleneck_stop_ids)}")
print(f"  Total trips analyzed: {len(bottleneck_trips)}")
